# 02 - Downstream QC

Combine all samples in one anndata object after filtering out low quality cells or doublets.

In [ ]:
from libraries import *
from parameters import *

In [ ]:
%load_ext rpy2.ipython

In [ ]:
os.getcwd()
os.chdir(projectDir)

***

## Load session

In [ ]:
with shelve.open('session_01.pkl', protocol=4, writeback=False) as db:
    for k in db.keys():
        globals()[k] = db[k]

***

## Filtering and cell calling

In [ ]:
for k, ad in tqdm(list(conf_samples.items())):
    if par_cutoff_min_genes: sc.pp.filter_cells(ad, min_genes=par_cutoff_min_genes)
    if par_cutoff_max_genes: sc.pp.filter_cells(ad, max_genes=par_cutoff_max_genes)
    if par_cutoff_min_counts: sc.pp.filter_cells(ad, min_counts=par_cutoff_min_counts)
    if par_final_empty_drops_fdr_cutoff: ad._inplace_subset_obs(ad.obs.empty_drops_FDR < par_final_empty_drops_fdr_cutoff)
    if par_mito_cutoff:
        if isinstance(par_mito_cutoff, str) and par_mito_cutoff.endswith('_'):
            ad._inplace_subset_obs(ad.obs.mt_frac < ad.obs[par_mito_cutoff])
        else:
            ad._inplace_subset_obs(ad.obs.mt_frac < par_mito_cutoff)
    if par_remove_mito_genes:
        ad._inplace_subset_var(~ad.var_names.str.startswith(conf_mt_prefix))

    display(ad)

In [ ]:
%%time

batch_categories, ads = zip(*conf_samples.items())

adata = sc.AnnData.concatenate(*ads, join=par_merge_type, batch_key=par_batch_key, batch_categories=batch_categories)
if 'X_diffmap' in adata.obsm_keys():
    del adata.obsm['X_diffmap']
del conf_samples

adata

In [ ]:
sc.pp.filter_genes(adata, min_cells=par_cutoff_min_cells)

In [ ]:
adata.X.shape

## Predict sex

In [ ]:
# conf_xist_gene_name = 'Xist' if par_species == 'mouse' else 'XIST'

# for adata in tqdm(list(conf_samples.values())):
#     if conf_xist_gene_name in adata.var_names:
#         adata.obs['predicted_sex'] = ['female' if x else 'male' for x in adata.obs_vector(conf_xist_gene_name) > 0]
#     else:
#         adata.obs['predicted_sex'] = 'male'

### Exclude mito genes and & sex genes

In [ ]:
# if par_remove_sex_genes:
#     from pyannotables import tables

#     for sample in tqdm(list(conf_samples.keys())):
#         genes = tables['mus_musculus-ensembl95-GRCm38'] if par_species == 'mouse' else tables['homo_sapiens-ensembl95-GRCh38']
#         sex_genes = genes.gene_name[genes.contig.isin(['X', 'Y'])].values

#         conf_samples[sample] = conf_samples[sample][:, ~conf_samples[sample].var_names.isin(sex_genes)].copy()
#         display(conf_samples[sample])

#     del tables

Save anndata object 

In [ ]:
adata.write(par_save_filename_1)